Architecture Overview

Agents
1. ReActAgent → main reasoning loop
2. ToolExecutor → async tool runner
3. Memory Module → shared state + trace
4. Tools → Functions agents can call
5. Controller → Orchestrates everything

LLM Interface → OpenAI or local model

1. Core System

In [11]:
import asyncio
import json
import uuid
from typing import Dict, Any, Callable, List, Optional
import aiohttp
import re

# =========================
# MEMORY
# =========================
class Memory:
    def __init__(self):
        self.history: List[Dict] = []
        self.state: Dict[str, Any] = {}

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})

    def last(self, n=5):
        return self.history[-n:]

    def set(self, key, value):
        self.state[key] = value

    def get(self, key):
        return self.state.get(key)


# =========================
# LLM INTERFACE
# =========================
class LLM:
    def __init__(self, backend="mock", api_key=None, model=None):
        self.backend = backend
        self.api_key = api_key
        self.model = model

    async def generate(self, prompt: str) -> str:
        if self.backend == "openai":
            from openai import AsyncOpenAI
            client = AsyncOpenAI(api_key=self.api_key)

            response = await client.chat.completions.create(
                model=self.model or "gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content

        elif self.backend == "local":
            # Example for Ollama or local server
            async with aiohttp.ClientSession() as session:
                async with session.post(
                    "http://localhost:11434/api/generate",
                    json={"model": self.model, "prompt": prompt}
                ) as resp:
                    data = await resp.json()
                    return data.get("response", "")

        else:
            # Mock reasoning (for testing)
            return self.mock_response(prompt)

    def mock_response(self, prompt):
        # Look for the last observation in the prompt history
        last_observation_match = re.search(r"Observation: (.*)", prompt)

        if "calculate" in prompt:
            if last_observation_match:
                # If there's an observation, it means the calculation is done, return final answer.
                result = last_observation_match.group(1).strip()
                return f"Final Answer: The calculated result is {result}"
            else:
                # First step for calculate task
                # Extract the expression from the user's task
                task_match = re.search(r"user: calculate (.*)", prompt)
                expression = task_match.group(1).strip() if task_match else "unknown expression"
                # Use json.dumps to correctly format the Action Input dictionary
                action_input_dict = {"expression": expression}
                action_input_str = json.dumps(action_input_dict)
                return f"Thought: I should use the calculator tool to evaluate the expression.\nAction: calculator\nAction Input: {action_input_str}"
        elif "search AI news and summarize" in prompt:
            if last_observation_match:
                if "Search results page:" in last_observation_match.group(1):
                    # After search, simulate summarize
                    return "Thought: I have the search results. Now I need to summarize them.\nAction: summarize\nAction Input: {\"text\": \"Simulated search results content for AI news...\"}"
                elif "..." in last_observation_match.group(1): # After summarize
                     return "Final Answer: AI news has been searched and summarized."
                else:
                    return "Final Answer: Done searching and summarizing AI news."
            else:
                # First step for search task
                return "Thought: I need to search for AI news.\nAction: search\nAction Input: {\"query\": \"AI news\"}"
        elif "fetch https://example.com and summarize" in prompt:
            if last_observation_match:
                # The actual fetch tool returns text[:1000]. Simulate this as an observation.
                if "<html" in last_observation_match.group(1) or "text[:1000]" in last_observation_match.group(1):
                    # After fetch, simulate summarize
                    return "Thought: I have fetched the URL content. Now I need to summarize it.\nAction: summarize\nAction Input: {\"text\": \"Simulated fetched content from example.com (first 1000 chars)...\"}"
                elif "..." in last_observation_match.group(1): # After summarize
                     return "Final Answer: The URL content has been fetched and summarized."
                else:
                    return "Final Answer: Done fetching and summarizing example.com."
            else:
                # First step for fetch task
                return "Thought: I need to fetch the content from the URL.\nAction: fetch\nAction Input: {\"url\": \"https://example.com\"}"

        return "Final Answer: Mock response for unhandled scenario."


/usr/local/lib/python3.12/dist-packages/IPython/lib/py311_tokenize.py:532: RuntimeWarning: coroutine 'main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)


Tool System (Async + Real APIs)

In [12]:
# =========================
# TOOL REGISTRY
# =========================
class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Callable] = {}

    def register(self, name: str, func: Callable):
        self.tools[name] = func

    async def execute(self, name: str, **kwargs):
        if name not in self.tools:
            raise ValueError(f"Tool {name} not found")
        return await self.tools[name](**kwargs)


# =========================
# TOOLS
# =========================

async def calculator(expression: str):
    try:
        return str(eval(expression))
    except Exception as e:
        return str(e)


async def web_search(query: str):
    url = f"https://duckduckgo.com/?q={query}&format=json"
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as resp:
            return f"Search results page: {query}"


async def fetch_url(url: str):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as resp:
            text = await resp.text()
            return text[:1000]


async def summarize(text: str):
    return text[:200] + "..."


async def parallel_tools(tools: List[Dict], registry: ToolRegistry):
    tasks = [
        registry.execute(t["name"], **t["args"])
        for t in tools
    ]
    return await asyncio.gather(*tasks)

3. ReAct Agent

In [3]:
# =========================
# REACT AGENT
# =========================
class ReActAgent:
    def __init__(self, llm: LLM, tools: ToolRegistry, memory: Memory):
        self.llm = llm
        self.tools = tools
        self.memory = memory

    async def run(self, task: str, max_steps=10):
        self.memory.add("user", task)

        for step in range(max_steps):
            prompt = self.build_prompt(task)

            response = await self.llm.generate(prompt)
            self.memory.add("assistant", response)

            parsed = self.parse_response(response)

            if parsed["type"] == "final":
                return parsed["content"]

            elif parsed["type"] == "action":
                result = await self.tools.execute(
                    parsed["action"],
                    **parsed["input"]
                )

                observation = f"Observation: {result}"
                self.memory.add("system", observation)

        return "Max steps reached"

    def build_prompt(self, task):
        history = "\n".join(
            [f"{h['role']}: {h['content']}" for h in self.memory.last(6)]
        )

        return f"""
You are a ReAct agent.

Task: {task}

{history}

Follow format:
Thought:
Action:
Action Input:

OR

Final Answer:
"""

    def parse_response(self, text: str):
        if "Final Answer:" in text:
            return {
                "type": "final",
                "content": text.split("Final Answer:")[-1].strip()
            }

        lines = text.split("\n")
        action = None
        action_input = {}

        for line in lines:
            if line.startswith("Action:"):
                action = line.replace("Action:", "").strip()
            if line.startswith("Action Input:"):
                try:
                    action_input = json.loads(
                        line.replace("Action Input:", "").strip()
                    )
                except:
                    action_input = {"input": line.split(":",1)[1].strip()}

        return {
            "type": "action",
            "action": action,
            "input": action_input
        }

4. Multi-Agent Parallel Controller

In [14]:
# =========================
# MULTI-AGENT ORCHESTRATOR
# =========================
class MultiAgentSystem:
    def __init__(self, llm: LLM):
        self.tools = ToolRegistry()
        self.llm = llm # Store LLM to pass to new agents

        # Register tools
        self.tools.register("calculator", calculator)
        self.tools.register("search", web_search)
        self.tools.register("fetch", fetch_url)
        self.tools.register("summarize", summarize)

    async def run(self, tasks: List[str]):
        # Create a new agent (and thus a new memory) for each task
        agents_for_tasks = [
            ReActAgent(self.llm, self.tools, Memory()) for _ in tasks
        ]
        results = await asyncio.gather(
            *[agent.run(task) for agent, task in zip(agents_for_tasks, tasks)]
        )
        return results

5. Testing Scenarios

In [15]:
async def main():
    llm = LLM(
        backend="mock",  # change to "openai" or "local"
        api_key="YOUR_KEY",
        model="gpt-4o-mini"
    )

    system = MultiAgentSystem(llm)

    tasks = [
        "calculate 5 * 7",
        "search AI news and summarize",
        "fetch https://example.com and summarize"
    ]

    results = await system.run(tasks)

    for t, r in zip(tasks, results):
        print(f"\nTASK: {t}")
        print(f"RESULT: {r}")


if __name__ == "__main__":
    # The line below is causing the RuntimeError in Colab because an event loop is already running.
    # Instead of asyncio.run(), we directly await main() since we are in an async environment.
    # asyncio.run(main())
    import nest_asyncio
    nest_asyncio.apply()
    await main()


TASK: calculate 5 * 7
RESULT: The calculated result is 35

TASK: search AI news and summarize
RESULT: AI news has been searched and summarized.

TASK: fetch https://example.com and summarize
RESULT: The URL content has been fetched and summarized.
